# LeetCode 1880 — Check if Word Equals Summation of Two Words bằng Python

Link đề: https://leetcode.com/problems/check-if-word-equals-summation-of-two-words/

Độ khó: **Easy** · Chủ đề: String

```python
class Solution:
    def isSumEqual(self, firstWord: str, secondWord: str, targetWord: str) -> bool:

```

**Mục tiêu học:**

1. Hiểu cách "đổi chữ thành chữ số" bằng `ord()` (mã ký tự) và phép trừ `ord(ch) - ord('a')`.
2. Cách trực giác: ghép các chữ số thành **chuỗi** rồi đổi sang `int` bằng `int("021")`.
3. Insight: không cần chuỗi trung gian — dùng công thức tích lũy `value = value * 10 + digit` (quy tắc Horner).
4. Một cách sáng tạo: `str.maketrans` + `translate` để dịch cả chuỗi chỉ trong một lệnh.
5. Trace từng bước, chứng minh tính đúng, phân tích Big-O và tự luyện biến thể (cơ số khác 10, chữ số khác `a..j`).

## 1. Bài toán yêu cầu gì?

- **Letter value** (giá trị chữ cái) = vị trí chữ cái trong bảng chữ cái, **bắt đầu từ 0**: `'a' -> 0`, `'b' -> 1`, ..., `'j' -> 9`.
- **Numerical value** của một chuỗi = **ghép** (concatenate) letter value của từng chữ rồi đổi sang số nguyên.
  - Ví dụ `"acb"` → `"0" + "2" + "1"` = `"021"` → `21`.
- Cho 3 chuỗi `firstWord`, `secondWord`, `targetWord` (chỉ gồm chữ `'a'` đến `'j'`).
- Trả `True` nếu `value(firstWord) + value(secondWord) == value(targetWord)`, ngược lại `False`.

| firstWord | secondWord | targetWord | Giá trị | Kết quả |
|---|---|---|---|:---:|
| `"acb"` | `"cba"` | `"cdb"` | `21 + 210 = 231` | `True` |
| `"aaa"` | `"a"` | `"aab"` | `0 + 0 ≠ 1` | `False` |
| `"aaa"` | `"a"` | `"aaaa"` | `0 + 0 = 0` | `True` |

**Ràng buộc:** độ dài mỗi chuỗi từ 1 đến 8, chỉ gồm `'a'..'j'`.

Vì chỉ có 10 chữ `a..j` ứng với đúng 10 chữ số `0..9`, mỗi chữ cái là **một chữ số thập phân**. Đây là chìa khóa của cả bài.

In [1]:
examples = [
    ("acb", "cba", "cdb", True),
    ("aaa", "a", "aab", False),
    ("aaa", "a", "aaaa", True),
]
for first, second, target, expected in examples:
    print(f"{first!r:7} + {second!r:7} =? {target!r:7} -> kỳ vọng {expected}")

'acb'   + 'cba'   =? 'cdb'   -> kỳ vọng True
'aaa'   + 'a'     =? 'aab'   -> kỳ vọng False
'aaa'   + 'a'     =? 'aaaa'  -> kỳ vọng True


## 2. Nền tảng Python cần biết

### 2.1. `ord()` và `chr()` — mã số của ký tự

Máy tính lưu mỗi ký tự bằng một con số (mã Unicode).

- `ord(ch)` trả về **mã số** của ký tự `ch`.
- `chr(code)` làm ngược lại: từ mã số ra ký tự.

Các chữ thường `'a'..'z'` có mã **liên tiếp**: `'a'` là 97, `'b'` là 98, ... Nên `ord(ch) - ord('a')` cho đúng vị trí bắt đầu từ 0.

In [2]:
for ch in "abcdj":
    print(ch, "-> ord =", ord(ch), "-> ord(ch) - ord('a') =", ord(ch) - ord("a"))

print(chr(97), chr(97 + 2))   # 'a', 'c'

a -> ord = 97 -> ord(ch) - ord('a') = 0
b -> ord = 98 -> ord(ch) - ord('a') = 1
c -> ord = 99 -> ord(ch) - ord('a') = 2
d -> ord = 100 -> ord(ch) - ord('a') = 3
j -> ord = 106 -> ord(ch) - ord('a') = 9
a c


**Lưu ý:** `ord('a')` là 97, **không phải** 0 hay 1. Nếu quên trừ `ord('a')` bạn sẽ được số rất lớn và sai hoàn toàn. Đây là lỗi kinh điển, ta sẽ gặp lại ở mục lỗi người mới.

### 2.2. Ghép chuỗi và đổi chuỗi chữ số sang `int`

- `"0" + "2"` cho `"02"` (ghép chuỗi, không phải cộng số).
- `str(5)` đổi số 5 thành chuỗi `"5"`.
- `int("021")` đổi chuỗi sang số nguyên → `21`. Python **tự bỏ số 0 ở đầu**, rất tiện cho bài này.

In [3]:
print("0" + "2" + "1")        # chuỗi '021'
print(int("021"))             # 21
print(int("000"))             # 0
print(int("0000") == int("0"))  # True
print(str(7) + str(3))        # '73'

021
21
0
True
73


### 2.3. `"".join(...)` và biểu thức generator

Ghép nhiều mảnh chuỗi bằng `+` trong vòng lặp thì dài dòng. Python có `"".join(danh_sach_chuoi)`:

- `"".join(["0", "2", "1"])` → `"021"`.
- Bên trong có thể viết **generator expression** `str(x) for x in ...` — giống list comprehension nhưng không tạo list trước.

In [4]:
pieces = ["0", "2", "1"]
print("".join(pieces))

word = "acb"
digits = [str(ord(ch) - ord("a")) for ch in word]   # list comprehension
print(digits)
print("".join(str(ord(ch) - ord("a")) for ch in word))  # generator, không cần []

021
['0', '2', '1']
021


### 2.4. Công thức tích lũy `value = value * 10 + d`

Khi đọc một số từ trái sang phải, mỗi lần gặp chữ số mới `d`, số cũ "dịch sang trái một hàng" (nhân 10) rồi cộng `d` vào hàng đơn vị.

Ví dụ đọc `2, 3, 1`:

```text
value = 0
gặp 2: value = 0 * 10 + 2 = 2
gặp 3: value = 2 * 10 + 3 = 23
gặp 1: value = 23 * 10 + 1 = 231
```

Công thức này gọi là **quy tắc Horner**; nó giống hệt cách `int("231")` hoạt động bên trong.

In [5]:
value = 0
for d in [2, 3, 1]:
    value = value * 10 + d
    print("gặp", d, "-> value =", value)

gặp 2 -> value = 2
gặp 3 -> value = 23
gặp 1 -> value = 231


### 2.5. Tính bằng tay trước khi code

Lấy ví dụ 1: `firstWord = "acb"`, `secondWord = "cba"`, `targetWord = "cdb"`.

| Chuỗi | Từng chữ → chữ số | Chuỗi chữ số | Số nguyên |
|---|---|---|---:|
| `acb` | a→0, c→2, b→1 | `"021"` | 21 |
| `cba` | c→2, b→1, a→0 | `"210"` | 210 |
| `cdb` | c→2, d→3, b→1 | `"231"` | 231 |

`21 + 210 = 231` → `True`.

Câu hỏi gợi mở:

- Tại sao `"aaa"` và `"a"` đều có giá trị 0? (vì toàn chữ số 0, số 0 ở đầu không có ý nghĩa).
- Có cần biết độ dài chuỗi bằng nhau không? (không — số 0 ở đầu tự mất khi đổi sang số).

In [6]:
def letter_value(ch: str) -> int:
    return ord(ch) - ord("a")

for word in ["acb", "cba", "cdb"]:
    print(word, "->", [letter_value(ch) for ch in word])

acb -> [0, 2, 1]
cba -> [2, 1, 0]
cdb -> [2, 3, 1]


## 3. Cách 1 — Ghép chuỗi chữ số rồi đổi sang `int` (trực giác)

Làm đúng như đề mô tả:

1. Với mỗi chữ `ch`, tính `ord(ch) - ord('a')`, đổi sang chuỗi.
2. Ghép lại thành chuỗi chữ số.
3. `int(...)` để có số nguyên.
4. So sánh tổng.

In [7]:
def word_value_string(word: str) -> int:
    digits = ""
    for ch in word:
        digits += str(ord(ch) - ord("a"))
    return int(digits)


def is_sum_equal_string(first: str, second: str, target: str) -> bool:
    return word_value_string(first) + word_value_string(second) == word_value_string(target)


for first, second, target, expected in examples:
    print(first, second, target, "->", is_sum_equal_string(first, second, target), "| kỳ vọng", expected)

acb cba cdb -> True | kỳ vọng True
aaa a aab -> False | kỳ vọng False
aaa a aaaa -> True | kỳ vọng True


### Code nộp LeetCode — Cách 1

In [8]:
class Solution:
    def isSumEqual(self, firstWord: str, secondWord: str, targetWord: str) -> bool:
        def value(word: str) -> int:
            return int("".join(str(ord(ch) - ord("a")) for ch in word))

        return value(firstWord) + value(secondWord) == value(targetWord)


sol = Solution()
print(sol.isSumEqual("acb", "cba", "cdb"))   # True
print(sol.isSumEqual("aaa", "a", "aab"))     # False

True
False


### Độ phức tạp của Cách 1

Gọi `L` = tổng độ dài ba chuỗi.

- Thời gian: `O(L)` — mỗi chữ cái được xử lý một lần (cộng thêm việc `int()` đọc chuỗi chữ số, cũng tuyến tính).
- Bộ nhớ: `O(L)` — phải tạo chuỗi chữ số trung gian.

**Nút thắt:** ta tạo chuỗi chỉ để rồi đổi ngay sang số. Với bài này chuỗi rất ngắn (≤ 8) nên không sao, nhưng về tư duy, ta đang làm "đường vòng": chữ → chuỗi → số. Có thể đi thẳng chữ → số không?

Thêm nữa, viết `digits += ...` trong vòng lặp tạo chuỗi mới mỗi lần (chuỗi Python bất biến) — thói quen không tốt khi chuỗi dài.

## 4. Bước đột phá — tích lũy trực tiếp bằng Horner

Vì mỗi chữ cái là **một chữ số thập phân**, numerical value của chuỗi chính là số thập phân có các chữ số đó. Mà ta đã biết công thức đọc số từ trái sang phải:

```python
value = value * 10 + (ord(ch) - ord('a'))
```

→ Không cần chuỗi trung gian, chỉ một biến `int`.

Kiểm tra nhanh với `"cdb"`: chữ số `2, 3, 1` → `0 → 2 → 23 → 231`. Đúng bằng `int("231")`.

In [9]:
value = 0
for ch in "cdb":
    d = ord(ch) - ord("a")
    value = value * 10 + d
print(value, "==", int("231"), "->", value == 231)

231 == 231 -> True


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — hàm chuyển một chuỗi thành số:**

```python
def word_value(word):
    value = 0                     # chưa đọc chữ nào → 0
    for ch in word:               # đọc từ trái sang phải
        d = ord(ch) - ord('a')    # chữ → chữ số 0..9
        value = value * 10 + d    # dịch trái 1 hàng rồi thêm d
    return value
```

**Đoạn 2 — so sánh:**

```python
return word_value(first) + word_value(second) == word_value(target)
```

Ghép lại:

In [10]:
def word_value(word: str) -> int:
    value = 0
    for ch in word:
        d = ord(ch) - ord("a")
        value = value * 10 + d
    return value


def is_sum_equal_horner(first: str, second: str, target: str) -> bool:
    return word_value(first) + word_value(second) == word_value(target)


for first, second, target, expected in examples:
    print(first, second, target, "->", is_sum_equal_horner(first, second, target), "| kỳ vọng", expected)

acb cba cdb -> True | kỳ vọng True
aaa a aab -> False | kỳ vọng False
aaa a aaaa -> True | kỳ vọng True


### Vì sao khởi tạo `value = 0`?

- Chưa đọc chữ nào thì giá trị là 0 — phần tử trung hòa của phép "nhân 10 rồi cộng".
- Nếu khởi tạo `value = 1` thì `"a"` sẽ ra `10` thay vì `0` — sai.

### Vì sao số 0 ở đầu không làm hỏng kết quả?

Với `"aab"`: `0 → 0*10+0 = 0 → 0*10+1 = 1`. Các số 0 đầu giữ `value = 0`, nhân 10 vẫn là 0. Giống hệt việc `int("001") == 1`.

In [11]:
for w in ["a", "aaa", "aab", "aaaa", "jjjjjjjj"]:
    print(f"{w!r:11} -> word_value = {word_value(w)}")

'a'         -> word_value = 0
'aaa'       -> word_value = 0
'aab'       -> word_value = 1
'aaaa'      -> word_value = 0
'jjjjjjjj'  -> word_value = 99999999


## 6. TRACE từng bước

Bảng trace cho `"cdb"` (target ví dụ 1):

| Bước | `ch` | `d = ord(ch)-97` | `value` trước | `value*10 + d` |
|---:|:---:|:---:|---:|---:|
| 1 | `c` | 2 | 0 | 2 |
| 2 | `d` | 3 | 2 | 23 |
| 3 | `b` | 1 | 23 | 231 |

Ý nghĩa các cột: `value trước` là số đã đọc được từ các chữ phía trước; cột cuối là giá trị **sau** khi thêm chữ hiện tại.

Hàm `trace_is_sum_equal` in mọi bước cho cả ba chuỗi rồi in phép so sánh cuối.

In [12]:
def trace_word_value(word: str) -> int:
    value = 0
    print(f"  Đọc {word!r}:")
    for step, ch in enumerate(word, start=1):
        d = ord(ch) - ord("a")
        new_value = value * 10 + d
        print(f"    bước {step}: ch={ch!r}, d={d}, value {value} -> {value}*10+{d} = {new_value}")
        value = new_value
    return value


def trace_is_sum_equal(first: str, second: str, target: str) -> bool:
    a = trace_word_value(first)
    b = trace_word_value(second)
    t = trace_word_value(target)
    ok = a + b == t
    print(f"  So sánh: {a} + {b} = {a + b} {'==' if ok else '!='} {t} -> {ok}")
    return ok


print("Ví dụ 1:")
trace_is_sum_equal("acb", "cba", "cdb")
print("Ví dụ 2:")
trace_is_sum_equal("aaa", "a", "aab")

Ví dụ 1:
  Đọc 'acb':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 2: ch='c', d=2, value 0 -> 0*10+2 = 2
    bước 3: ch='b', d=1, value 2 -> 2*10+1 = 21
  Đọc 'cba':
    bước 1: ch='c', d=2, value 0 -> 0*10+2 = 2
    bước 2: ch='b', d=1, value 2 -> 2*10+1 = 21
    bước 3: ch='a', d=0, value 21 -> 21*10+0 = 210
  Đọc 'cdb':
    bước 1: ch='c', d=2, value 0 -> 0*10+2 = 2
    bước 2: ch='d', d=3, value 2 -> 2*10+3 = 23
    bước 3: ch='b', d=1, value 23 -> 23*10+1 = 231
  So sánh: 21 + 210 = 231 == 231 -> True
Ví dụ 2:
  Đọc 'aaa':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 2: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 3: ch='a', d=0, value 0 -> 0*10+0 = 0
  Đọc 'a':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
  Đọc 'aab':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 2: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 3: ch='b', d=1, value 0 -> 0*10+1 = 1
  So sánh: 0 + 0 = 0 != 1 -> False


False

### Một nhánh đáng chú ý: các chuỗi khác độ dài

Ví dụ 3 có `targetWord = "aaaa"` dài 4 còn hai chuỗi kia dài 3 và 1. Trace cho thấy chuỗi dài hơn không làm sai gì vì toàn số 0 đầu.

In [13]:
print("Ví dụ 3:")
trace_is_sum_equal("aaa", "a", "aaaa")
print("Có nhớ (carry):")
trace_is_sum_equal("j", "b", "ba")   # 9 + 1 = 10

Ví dụ 3:
  Đọc 'aaa':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 2: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 3: ch='a', d=0, value 0 -> 0*10+0 = 0
  Đọc 'a':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
  Đọc 'aaaa':
    bước 1: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 2: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 3: ch='a', d=0, value 0 -> 0*10+0 = 0
    bước 4: ch='a', d=0, value 0 -> 0*10+0 = 0
  So sánh: 0 + 0 = 0 == 0 -> True
Có nhớ (carry):
  Đọc 'j':
    bước 1: ch='j', d=9, value 0 -> 0*10+9 = 9
  Đọc 'b':
    bước 1: ch='b', d=1, value 0 -> 0*10+1 = 1
  Đọc 'ba':
    bước 1: ch='b', d=1, value 0 -> 0*10+1 = 1
    bước 2: ch='a', d=0, value 1 -> 1*10+0 = 10
  So sánh: 9 + 1 = 10 == 10 -> True


True

## 7. Cách 3 — `str.maketrans` + `translate` (sáng tạo, rất Pythonic)

`str.maketrans(src, dst)` tạo một **bảng dịch** ký tự: ký tự thứ `i` của `src` được thay bằng ký tự thứ `i` của `dst`. Sau đó `word.translate(table)` thay **mọi** ký tự cùng lúc.

Bảng dịch của bài: `"abcdefghij"` → `"0123456789"`.

In [14]:
table = str.maketrans("abcdefghij", "0123456789")
print(table)                     # dict mã ký tự -> mã ký tự
print("acb".translate(table))    # '021'
print(int("cdb".translate(table)))

{97: 48, 98: 49, 99: 50, 100: 51, 101: 52, 102: 53, 103: 54, 104: 55, 105: 56, 106: 57}
021
231


In [15]:
def is_sum_equal_translate(first: str, second: str, target: str) -> bool:
    table = str.maketrans("abcdefghij", "0123456789")
    to_int = lambda w: int(w.translate(table))
    return to_int(first) + to_int(second) == to_int(target)


for first, second, target, expected in examples:
    print(first, second, target, "->", is_sum_equal_translate(first, second, target))

acb cba cdb -> True
aaa a aab -> False
aaa a aaaa -> True


`lambda w: ...` là cách viết **hàm vô danh** một dòng: `to_int = lambda w: int(w.translate(table))` tương đương

```python
def to_int(w):
    return int(w.translate(table))
```

### Code nộp LeetCode — Cách 3

In [16]:
class Solution:
    def isSumEqual(self, firstWord: str, secondWord: str, targetWord: str) -> bool:
        table = str.maketrans("abcdefghij", "0123456789")
        f = lambda w: int(w.translate(table))
        return f(firstWord) + f(secondWord) == f(targetWord)


print(Solution().isSumEqual("aaa", "a", "aaaa"))  # True

True


## 8. Code nộp LeetCode — lời giải chính (Horner)

Bản gọn để submit, không print.

In [17]:
class Solution:
    def isSumEqual(self, firstWord: str, secondWord: str, targetWord: str) -> bool:
        def value(word: str) -> int:
            v = 0
            for ch in word:
                v = v * 10 + (ord(ch) - ord("a"))
            return v

        return value(firstWord) + value(secondWord) == value(targetWord)


sol = Solution()
for first, second, target, expected in examples:
    assert sol.isSumEqual(first, second, target) == expected
print("Solution (Horner) PASS 3 ví dụ của đề")

Solution (Horner) PASS 3 ví dụ của đề


## 9. Vì sao thuật toán đúng?

**Invariant (bất biến) của vòng lặp Horner:** sau khi đọc `k` chữ đầu của `word`, `value` bằng đúng numerical value của tiền tố `word[:k]`.

- **Khởi đầu** (`k = 0`): tiền tố rỗng, `value = 0`. Đúng theo quy ước.
- **Duy trì:** giả sử `value = num(word[:k])`. Ghép thêm chữ số `d` vào bên phải một số thập phân thì giá trị mới là `num * 10 + d` (mọi chữ số cũ lên một hàng). Đó chính là phép cập nhật.
- **Kết thúc** (`k = len(word)`): `value = num(word)`.

Điều kiện "chỉ có `a..j`" đảm bảo `d` luôn trong `0..9` — tức là **một** chữ số thập phân. Nếu có `'k'` (giá trị 10), phép ghép `"10"` thêm **hai** chữ số và công thức `*10` sẽ sai (xem mục biến thể).

Phép so sánh cuối chính là định nghĩa của đề, nên kết quả đúng.

## 10. Độ phức tạp và so sánh các cách

Gọi `L = len(firstWord) + len(secondWord) + len(targetWord)` (ở đây `L ≤ 24`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|:---:|:---:|---|
| 1. Ghép chuỗi + `int()` | làm đúng theo mô tả đề | `O(L)` | `O(L)` | dễ nghĩ nhất |
| 2. Horner | `v = v*10 + d` | `O(L)` | `O(1)` | không cần chuỗi trung gian, dùng được ở mọi ngôn ngữ |
| 3. `maketrans/translate` | bảng dịch ký tự | `O(L)` | `O(L)` | ngắn, Pythonic |

Ghi chú: số lớn nhất là `"jjjjjjjj"` = 99.999.999, vẫn nằm gọn trong `int` 32-bit, nên ở C++/Java cũng không lo tràn số.

## 11. Bộ test kiểm tra

Nhóm test:

1. Ba ví dụ của đề.
2. Biên: chuỗi 1 ký tự (`"a"`, `"j"`), chuỗi dài 8 toàn `'j'`.
3. Có nhớ (carry): `"j" + "b" = "ba"` (9 + 1 = 10).
4. Phản ví dụ dễ sai: số 0 ở đầu target (`"ab"` có giá trị 1).
5. Đối chiếu 3 cách trên nhiều input sinh ngẫu nhiên (có seed).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:32} PASS")


solvers = [is_sum_equal_string, is_sum_equal_horner, is_sum_equal_translate]

tests = [
    ("ví dụ 1", ("acb", "cba", "cdb"), True),
    ("ví dụ 2", ("aaa", "a", "aab"), False),
    ("ví dụ 3", ("aaa", "a", "aaaa"), True),
    ("1 ký tự a+a=a", ("a", "a", "a"), True),
    ("9 + 0 = 9", ("j", "a", "j"), True),
    ("carry 9+1=10", ("j", "b", "ba"), True),
    ("số 0 đầu target", ("a", "b", "ab"), True),
    ("max + 0", ("jjjjjjjj", "a", "jjjjjjjj"), True),
    ("max + 1 sai", ("jjjjjjjj", "b", "jjjjjjjj"), False),
]
for name, args, expected in tests:
    for f in solvers:
        check(f"{name} [{f.__name__[13:]}]", f(*args), expected)

ví dụ 1 [string]                 PASS
ví dụ 1 [horner]                 PASS
ví dụ 1 [translate]              PASS
ví dụ 2 [string]                 PASS
ví dụ 2 [horner]                 PASS
ví dụ 2 [translate]              PASS
ví dụ 3 [string]                 PASS
ví dụ 3 [horner]                 PASS
ví dụ 3 [translate]              PASS
1 ký tự a+a=a [string]           PASS
1 ký tự a+a=a [horner]           PASS
1 ký tự a+a=a [translate]        PASS
9 + 0 = 9 [string]               PASS
9 + 0 = 9 [horner]               PASS
9 + 0 = 9 [translate]            PASS
carry 9+1=10 [string]            PASS
carry 9+1=10 [horner]            PASS
carry 9+1=10 [translate]         PASS
số 0 đầu target [string]         PASS
số 0 đầu target [horner]         PASS
số 0 đầu target [translate]      PASS
max + 0 [string]                 PASS
max + 0 [horner]                 PASS
max + 0 [translate]              PASS
max + 1 sai [string]             PASS
max + 1 sai [horner]             PASS
max + 1 sai 

In [19]:
import random

random.seed(1880)
letters = "abcdefghij"

def rand_word():
    return "".join(random.choice(letters) for _ in range(random.randint(1, 8)))

mismatch = 0
for _ in range(2000):
    a, b, t = rand_word(), rand_word(), rand_word()
    if random.random() < 0.3:
        # tạo target đúng bằng tổng để có nhiều ca True
        s = str(word_value(a) + word_value(b))
        if len(s) <= 8:
            t = "".join(letters[int(x)] for x in s)
    results = {f(a, b, t) for f in solvers}
    if len(results) != 1:
        mismatch += 1
print("Số ca 3 cách cho kết quả khác nhau:", mismatch)
assert mismatch == 0

Số ca 3 cách cho kết quả khác nhau: 0


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên trừ `ord('a')`

```python
v = v * 10 + ord(ch)       # SAI: ord('a') = 97
```

Vì sao sai: mỗi "chữ số" thành 97..106, phép `*10` không còn là ghép chữ số. Sửa: `ord(ch) - ord('a')`.

In [20]:
def wrong_value_no_offset(word):
    v = 0
    for ch in word:
        v = v * 10 + ord(ch)
    return v

print("Sai:", wrong_value_no_offset("acb"), "| Đúng:", word_value("acb"))

Sai: 10788 | Đúng: 21


### Lỗi 2: Cộng chữ số thay vì ghép

```python
v += ord(ch) - ord('a')    # SAI: tính tổng các chữ số
```

`"acb"` sẽ ra `0 + 2 + 1 = 3` thay vì `21`. Đề nói **concatenate** (ghép), không phải cộng.

### Lỗi 3: Cộng chuỗi thay vì cộng số

```python
return s1 + s2 == s3       # SAI nếu s1, s2, s3 là chuỗi chữ số
```

`"021" + "210"` là `"021210"` (ghép chuỗi), không phải 231. Phải `int()` trước khi cộng.

In [21]:
def wrong_sum_digits(word):
    v = 0
    for ch in word:
        v += ord(ch) - ord("a")
    return v

print("Lỗi 2:", wrong_sum_digits("acb"), "thay vì", word_value("acb"))

s1, s2, s3 = "021", "210", "231"
print("Lỗi 3:", repr(s1 + s2), "==", repr(s3), "->", s1 + s2 == s3)
print("Sửa  :", int(s1) + int(s2), "==", int(s3), "->", int(s1) + int(s2) == int(s3))

Lỗi 2: 3 thay vì 21
Lỗi 3: '021210' == '231' -> False
Sửa  : 231 == 231 -> True


### Lỗi 4: So sánh chuỗi chữ số thay vì số

```python
return str(a + b) == targetDigits   # SAI khi target có số 0 ở đầu
```

Với `"a" + "b"` vs `"ab"`: `str(0 + 1) = "1"` nhưng chuỗi chữ số của target là `"01"` → so chuỗi ra `False`, trong khi đáp án đúng là `True`. Luôn so sánh **số**.

In [22]:
target_digits = "ab".translate(str.maketrans("abcdefghij", "0123456789"))
print("Chuỗi:", repr(str(0 + 1)), "==", repr(target_digits), "->", str(0 + 1) == target_digits)
print("Số   :", 0 + 1, "==", int(target_digits), "->", 0 + 1 == int(target_digits))

Chuỗi: '1' == '01' -> False
Số   : 1 == 1 -> True


## 13. Mẹo nhận dạng pattern

- Thấy "**ghép các chữ số rồi đổi thành số**" → nghĩ ngay tới `value = value * base + digit` (Horner).
- Thấy "**chữ cái ↔ vị trí trong bảng chữ cái**" → `ord(ch) - ord('a')` (hoặc `+1` nếu đếm từ 1, như LeetCode 171 Excel Column).
- Bài nhỏ, chuỗi ngắn → cách "làm đúng như đề" (ghép chuỗi + `int`) hoàn toàn chấp nhận được; nhưng hiểu Horner giúp giải các bài cơ số khác, chuỗi siêu dài, hay ngôn ngữ không có `int()` thông minh.

Các bài liên quan: 171 (Excel Sheet Column Number), 1281, 415 (Add Strings), 2: Add Two Numbers, 989 (Add to Array-Form).

## 14. Biến thể

### Biến thể 1: chữ cái `a..z` (giá trị 0..25) và quy tắc "ghép"

Nếu cho phép `'k'` (10), ghép `"b" + "k"` là `"1" + "10"` = `"110"`. Lúc này một chữ cái có thể là **2 chữ số**, nên công thức `*10` không đúng nữa — phải nhân với `10 ** (số chữ số của d)`.

In [23]:
def value_concat_any(word: str) -> int:
    v = 0
    for ch in word:
        d = ord(ch) - ord("a")
        v = v * (10 ** len(str(d))) + d
    return v

print(value_concat_any("bk"), "==", int("1" + "10"))
print(value_concat_any("acb"), "==", word_value("acb"))

110 == 110
21 == 21


### Biến thể 2: mỗi chữ cái là một chữ số trong **cơ số 26**

Nếu đề nói "chuỗi là số trong cơ số 26 với `a=0..z=25`", chỉ cần đổi `10` thành `26`. Đây là cùng một khung Horner.

In [24]:
def value_base(word: str, base: int) -> int:
    v = 0
    for ch in word:
        v = v * base + (ord(ch) - ord("a"))
    return v

print(value_base("ba", 26))   # 1*26 + 0 = 26
print(value_base("cdb", 10))  # 231

26
231


## 15. Bài tập tự luyện (TODO)

Làm theo thứ tự, mỗi bài khó hơn bài trước. **Không** chạy test khi chưa viết xong (test để trong comment).

### Bài 1 — Dự đoán output (tính tay)

Không chạy code, hãy cho biết numerical value của: `"j"`, `"ba"`, `"abc"`, `"jab"`, `"aaaab"`. Viết kết quả vào list `ex1_answers`.

In [25]:
# TODO: điền 5 số nguyên theo thứ tự "j", "ba", "abc", "jab", "aaaab"
ex1_answers = None

# Kiểm tra sau khi điền (bỏ comment):
# assert ex1_answers == [word_value(w) for w in ["j", "ba", "abc", "jab", "aaaab"]]

### Bài 2 — Điền chỗ trống: hàm `word_value` theo Horner

In [26]:
def ex2_word_value(word: str) -> int:
    v = 0
    for ch in word:
        # TODO: tính d từ ch
        # TODO: cập nhật v
        pass
    return v

# assert ex2_word_value("cdb") == 231
# assert ex2_word_value("aaa") == 0

### Bài 3 — Đổi ngược: từ số nguyên ra chuỗi `a..j`

Viết `number_to_word(x)` trả về chuỗi **ngắn nhất** có numerical value bằng `x` (`x >= 0`). Ví dụ `231 -> "cdb"`, `0 -> "a"`, `10 -> "ba"`.

In [27]:
def number_to_word(x: int) -> str:
    # TODO: lấy từng chữ số của x rồi đổi sang chữ cái
    pass

# assert number_to_word(231) == "cdb"
# assert number_to_word(0) == "a"
# assert number_to_word(10) == "ba"

### Bài 4 — Tổng của nhiều từ

Cho list `words` và `target`. Trả `True` nếu tổng numerical value của mọi từ trong `words` bằng numerical value của `target`.

In [28]:
def is_sum_equal_many(words: list[str], target: str) -> bool:
    # TODO
    pass

# assert is_sum_equal_many(["acb", "cba"], "cdb") == True
# assert is_sum_equal_many(["b", "b", "b"], "d") == True
# assert is_sum_equal_many(["j", "j"], "bi") == True    # 9 + 9 = 18
# assert is_sum_equal_many(["j"], "a") == False

### Bài 5 — Cộng không đổi sang số (mini challenge)

Viết `add_words(a, b)` trả về **chuỗi chữ cái** biểu diễn tổng, cộng từng chữ số từ phải sang trái kèm biến nhớ `carry` (giống Add Strings — LeetCode 415). Không được gọi `int()` trên cả chuỗi. Kết quả không có `'a'` thừa ở đầu (trừ khi tổng bằng 0 → `"a"`).

In [29]:
def add_words(a: str, b: str) -> str:
    # TODO: i, j chạy từ cuối; carry; nối kết quả rồi đảo ngược
    pass

# assert add_words("acb", "cba") == "cdb"
# assert add_words("j", "b") == "ba"
# assert add_words("aaa", "a") == "a"

### Bài 6 — Biến thể cơ số 26

Viết `is_sum_equal_base26(a, b, t)`: cùng đề nhưng mỗi chữ `a..z` là một chữ số trong cơ số 26.

In [30]:
def is_sum_equal_base26(a: str, b: str, t: str) -> bool:
    # TODO
    pass

# assert is_sum_equal_base26("z", "b", "ba") == True   # 25 + 1 = 26 = "ba"
# assert is_sum_equal_base26("z", "a", "z") == True

## 16. Gợi ý 3 mức

**Bài 1.** (1) Đổi từng chữ thành chữ số rồi đọc như số thường. (2) `j=9`, `b=1`. (3) `"jab"` → `"901"` → 901.

**Bài 2.** (1) Mỗi chữ là một chữ số. (2) `d = ord(ch) - ord('a')`. (3) `v = v * 10 + d`.

**Bài 3.** (1) Lấy chữ số cuối bằng `% 10`, bỏ bằng `// 10`. (2) Chữ cái là `chr(ord('a') + digit)`. (3) Xử lý riêng `x == 0`; gom chữ cái vào list rồi đảo ngược, `"".join`.

**Bài 4.** (1) Dùng lại `word_value`. (2) `sum(...)` với generator. (3) `sum(word_value(w) for w in words) == word_value(target)`.

**Bài 5.** (1) Như cộng tay ở tiểu học. (2) `i = len(a)-1`, `j = len(b)-1`, lặp khi `i >= 0 or j >= 0 or carry`. (3) `s = da + db + carry; out.append(chr(97 + s % 10)); carry = s // 10`; cuối cùng bỏ `'a'` thừa ở đầu.

**Bài 6.** (1) Horner với base 26. (2) Viết hàm `value_base(word, 26)`. (3) So sánh tổng.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [31]:
ans_ex1 = [9, 10, 12, 901, 1]


def ans_ex2_word_value(word: str) -> int:
    v = 0
    for ch in word:
        d = ord(ch) - ord("a")
        v = v * 10 + d
    return v


def ans_number_to_word(x: int) -> str:
    if x == 0:
        return "a"
    out = []
    while x > 0:
        out.append(chr(ord("a") + x % 10))
        x //= 10
    return "".join(reversed(out))


def ans_is_sum_equal_many(words, target):
    return sum(ans_ex2_word_value(w) for w in words) == ans_ex2_word_value(target)


def ans_add_words(a: str, b: str) -> str:
    i, j, carry = len(a) - 1, len(b) - 1, 0
    out = []
    while i >= 0 or j >= 0 or carry:
        da = ord(a[i]) - 97 if i >= 0 else 0
        db = ord(b[j]) - 97 if j >= 0 else 0
        s = da + db + carry
        out.append(chr(97 + s % 10))
        carry = s // 10
        i -= 1
        j -= 1
    res = "".join(reversed(out)).lstrip("a")
    return res or "a"


def ans_is_sum_equal_base26(a, b, t):
    def val(w):
        v = 0
        for ch in w:
            v = v * 26 + (ord(ch) - 97)
        return v
    return val(a) + val(b) == val(t)


assert ans_ex1 == [word_value(w) for w in ["j", "ba", "abc", "jab", "aaaab"]]
assert ans_ex2_word_value("cdb") == 231 and ans_ex2_word_value("aaa") == 0
assert ans_number_to_word(231) == "cdb" and ans_number_to_word(0) == "a" and ans_number_to_word(10) == "ba"
assert ans_is_sum_equal_many(["acb", "cba"], "cdb") and ans_is_sum_equal_many(["b", "b", "b"], "d")
assert ans_is_sum_equal_many(["j", "j"], "bi") and not ans_is_sum_equal_many(["j"], "a")
assert ans_add_words("acb", "cba") == "cdb" and ans_add_words("j", "b") == "ba" and ans_add_words("aaa", "a") == "a"
assert ans_is_sum_equal_base26("z", "b", "ba") and ans_is_sum_equal_base26("z", "a", "z")

# kiểm tra chéo add_words với cách dùng số
random.seed(7)
for _ in range(500):
    a, b = rand_word(), rand_word()
    assert word_value(ans_add_words(a, b)) == word_value(a) + word_value(b)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isSumEqual(self, firstWord: str, secondWord: str, targetWord: str) -> bool:
        def value(word):
            v = 0
            for ch in word:
                v = v * 10 + (ord(ch) - ord('a'))
            return v
        return value(firstWord) + value(secondWord) == value(targetWord)
```

**Khung tư duy:** chữ → chữ số (`ord - ord('a')`) → đọc số từ trái sang phải (`v*10 + d`) → so sánh.

**Checklist trước khi submit:**

- [ ] Đã trừ `ord('a')` chưa?
- [ ] Khởi tạo `v = 0` (không phải 1)?
- [ ] Đang **ghép** chữ số (`*10`) chứ không phải cộng dồn chữ số?
- [ ] So sánh **số nguyên**, không so chuỗi chữ số (tránh lỗi số 0 đầu)?
- [ ] Tên method đúng `isSumEqual`, trả về `bool`?